# 잠실역 지하철 2024 추출

In [ ]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

source_rel_path = Path("Data/Metro/서울교통공사_역별 시간대별 승하차인원(24.1~24.12) (1).csv")
encodings = ["cp949", "euc-kr", "utf-8-sig"]
target_station = "잠실역"
target_lines = {"2호선", "8호선"}

project_root = Path.cwd().resolve()
for candidate_root in [project_root, project_root.parent]:
    candidate_source = candidate_root / source_rel_path
    if candidate_source.exists():
        project_root = candidate_root
        source_path = candidate_source
        break
else:
    raise FileNotFoundError(f"Source CSV not found: {source_rel_path}")

output_path = project_root / "Data/Metro/jamsil_metro_2024.csv"


def normalize_col(name):
    return re.sub(r"\s+", "", str(name)).strip()


def normalize_station(name):
    text = str(name).strip()
    text = re.sub(r"\s+", "", text)
    text = text.replace("역", "")
    text = re.sub(r"\([^)]*\)", "", text)
    return text


def pick_column(columns, keywords):
    normalized = {col: normalize_col(col) for col in columns}
    for keyword in keywords:
        for col, norm in normalized.items():
            if keyword in norm:
                return col
    return None


def pick_hour_columns(columns):
    normalized = {col: normalize_col(col) for col in columns}
    selected = []
    for hour in range(6, 24):
        hour_token = f"{hour:02d}시"
        next_token = f"{hour + 1:02d}시"
        candidates = [
            col for col, norm in normalized.items()
            if norm.startswith(hour_token)
            and "이전" not in norm
            and "이후" not in norm
        ]
        if not candidates:
            candidates = [
                col for col, norm in normalized.items()
                if norm.startswith(f"{hour}시")
                and "이전" not in norm
                and "이후" not in norm
            ]
        if not candidates:
            raise KeyError(f"Hour column not found for {hour_token}")
        exact = [col for col in candidates if next_token in normalize_col(col)]
        selected.append(exact[0] if exact else candidates[0])
    return selected


last_error = None
for enc in encodings:
    try:
        df = pd.read_csv(source_path, encoding=enc)
        used_encoding = enc
        break
    except UnicodeDecodeError as exc:
        last_error = exc
else:
    raise RuntimeError(f"Failed to read source CSV with expected encodings: {last_error}")

station_col = pick_column(df.columns, ["역명", "역"])
line_col = pick_column(df.columns, ["호선"])
date_col = pick_column(df.columns, ["수송일자", "날짜"])
direction_col = pick_column(df.columns, ["승하차구분", "구분"])
hour_cols = pick_hour_columns(df.columns)

df[date_col] = pd.to_datetime(df[date_col])
station_key = normalize_station(target_station)
line_keys = {normalize_col(line) for line in target_lines}

filtered = df[
    df[station_col].map(normalize_station).eq(station_key)
    & df[line_col].astype(str).map(lambda x: normalize_col(x) in line_keys)
    & df[date_col].dt.year.eq(2024)
].copy()

for col in hour_cols:
    filtered[col] = pd.to_numeric(filtered[col], errors="coerce").fillna(0)

group_cols = [date_col, line_col, station_col]
if direction_col is not None and direction_col not in group_cols:
    group_cols.append(direction_col)

result_df = (
    filtered.groupby(group_cols, as_index=False)[hour_cols]
    .sum()
    .sort_values(group_cols)
    .reset_index(drop=True)
)

rename_map = {}
for col in group_cols:
    norm = normalize_col(col)
    if "날짜" in norm or "수송일자" in norm:
        rename_map[col] = "날짜"
    elif "호선" in norm:
        rename_map[col] = "호선"
    elif "역" in norm:
        rename_map[col] = "역명"
    elif "구분" in norm:
        rename_map[col] = "구분"

result_df = result_df.rename(columns=rename_map)
if "날짜" in result_df.columns:
    result_df["날짜"] = pd.to_datetime(result_df["날짜"]).dt.strftime("%Y-%m-%d")

metadata_cols = [col for col in ["날짜", "호선", "역명", "구분"] if col in result_df.columns]
result_df = result_df[metadata_cols + hour_cols]
result_df.to_csv(output_path, index=False)

display(result_df.head())
print(result_df.shape)
print(result_df.columns.tolist())
print(f"encoding used: {used_encoding}")
print(f"source: {source_path}")
print(f"saved: {output_path}")